# Specimen 02 — ReAct-Style Loop

Goal: turn a single tool call into a real loop — reason, act, observe, repeat — so the model can chain multiple tool calls toward a multi-step goal.

In [1]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'


## 1. Reuse specimen 01's tools inside a loop

Keep calling the API until `stop_reason` is no longer `tool_use`.

In [2]:
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression involving addition, subtraction, multiplication, and division of numbers.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "A basic arithmetic expression, e.g. '482391 * 17038'"}},
        "required": ["expression"]
    }
}

def run_calculator(expression):
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        raise ValueError(f'Unsupported characters in expression: {expression}')
    if len(expression) > 100:
        raise ValueError('Expression too long')
    if '**' in expression:
        raise ValueError('Exponentiation is not supported')
    return eval(expression, {"__builtins__": {}}, {})

local_dataset = {
    "price_widget_a": 42.50,
    "price_widget_b": 17.25,
    "price_widget_c": 8.00,
}

lookup_tool = {
    "name": "lookup_price",
    "description": "Look up the unit price of a product by its key from a small local dataset. Use this instead of guessing prices.",
    "input_schema": {
        "type": "object",
        "properties": {"key": {"type": "string", "description": "Product key, e.g. 'price_widget_a'"}},
        "required": ["key"]
    }
}

def run_lookup(key):
    if key not in local_dataset:
        raise KeyError(f"No price found for '{key}'")
    return local_dataset[key]

TOOLS = [calculator_tool, lookup_tool]
TOOL_FUNCTIONS = {
    "calculator": lambda inp: run_calculator(inp['expression']),
    "lookup_price": lambda inp: run_lookup(inp['key']),
}

def react_loop_basic(user_message, tools, tool_functions, max_tokens=500, max_steps=10):
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
    step = 1

    while response.stop_reason == 'tool_use':
        if step >= max_steps:
            raise RuntimeError(f'Hit max_steps={max_steps} without finishing')
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                result = tool_functions[block.name](block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
        step += 1

    return ''.join(b.text for b in response.content if b.type == 'text')

print(react_loop_basic("What is 15 times 7?", TOOLS, TOOL_FUNCTIONS))

105


## 2. Give it a task needing 2+ tool calls

e.g. 'look up X, then compute Y using what you found.' Confirm it calls the lookup tool rather than hallucinating the value.

In [3]:
answer = react_loop_basic(
    "Look up the price of widget_a and widget_b, then tell me the total cost of buying 3 of widget_a and 2 of widget_b.",
    TOOLS, TOOL_FUNCTIONS,
)
print(answer)

expected_total = 3 * local_dataset['price_widget_a'] + 2 * local_dataset['price_widget_b']
print(f'\nExpected total (computed independently): {expected_total}')

Here's the breakdown:

| Item | Unit Price | Qty | Subtotal |
|---|---|---|---|
| widget_a | $42.50 | 3 | $127.50 |
| widget_b | $17.25 | 2 | $34.50 |
| **Total** | | | **$162.00** |

Buying 3 of widget_a and 2 of widget_b costs **$162.00**.

Expected total (computed independently): 162.0


## 3. Log every step

Print each reasoning/tool_use/observation trio as it happens — watch the loop reason, act, and observe in real time.

In [4]:
def react_loop_logged(user_message, tools, tool_functions, max_tokens=500, max_steps=10):
    messages = [{"role": "user", "content": user_message}]
    step = 0

    while True:
        step += 1
        if step > max_steps:
            raise RuntimeError(f'Hit max_steps={max_steps} without finishing')
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})

        for block in response.content:
            if block.type == 'text' and block.text.strip():
                print(f'[step {step}] REASON: {block.text.strip()}')

        if response.stop_reason != 'tool_use':
            final_text = ''.join(b.text for b in response.content if b.type == 'text')
            print(f'[step {step}] FINAL: {final_text}')
            return final_text

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                print(f'[step {step}] ACT: {block.name}({block.input})')
                result = tool_functions[block.name](block.input)
                print(f'[step {step}] OBSERVE: {result}')
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})

react_loop_logged(
    "Look up the price of widget_a and widget_c, then tell me which one is cheaper and by how much.",
    TOOLS, TOOL_FUNCTIONS,
)

[step 1] REASON: I'll look up both prices.
[step 1] ACT: lookup_price({'key': 'price_widget_a'})
[step 1] OBSERVE: 42.5
[step 1] ACT: lookup_price({'key': 'price_widget_c'})
[step 1] OBSERVE: 8.0


[step 2] ACT: calculator({'expression': '42.5 - 8.0'})
[step 2] OBSERVE: 34.5


[step 3] REASON: Here's what I found:

- **widget_a:** 42.50
- **widget_c:** 8.00

**widget_c is cheaper by 34.50** — roughly one-fifth the price of widget_a.
[step 3] FINAL: Here's what I found:

- **widget_a:** 42.50
- **widget_c:** 8.00

**widget_c is cheaper by 34.50** — roughly one-fifth the price of widget_a.


"Here's what I found:\n\n- **widget_a:** 42.50\n- **widget_c:** 8.00\n\n**widget_c is cheaper by 34.50** — roughly one-fifth the price of widget_a."

## 4. Add a step cap

A max-iteration count that breaks the loop with a clear message instead of running forever. Test it actually triggers on a task that can't terminate.

In [5]:
def react_loop_capped(user_message, tools, tool_functions, max_tokens=500, max_steps=5):
    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})

        if response.stop_reason != 'tool_use':
            return ''.join(b.text for b in response.content if b.type == 'text'), step

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                result = tool_functions[block.name](block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})

    return None, step

unsolvable_tool = {
    "name": "increment_counter",
    "description": "Increment an internal counter by 1 and return the new value. Calling this repeatedly never produces a final answer on its own.",
    "input_schema": {"type": "object", "properties": {}, "required": []}
}

counter_state = {"value": 0}
def run_increment(_inp):
    counter_state["value"] += 1
    return counter_state["value"]

trap_tools = [unsolvable_tool]
trap_functions = {"increment_counter": run_increment}

answer, steps_used = react_loop_capped(
    "Keep incrementing the counter until it reaches exactly 1000000, then tell me the final value.",
    trap_tools, trap_functions, max_steps=5,
)

print(f'Steps used: {steps_used}')
print(f'Answer: {answer}')
print(f'Cap triggered: {answer is None and steps_used == 5}')

Steps used: 1
Answer: I need to flag a problem with this request before doing anything.

Reaching 1,000,000 would require a million sequential tool calls. Each one is a separate round trip, and I can only issue a bounded number per turn — realistically I'd exhaust the context window after a few thousand at most, and even in a best case this would take an enormous amount of time and compute. The task can't complete as specified. Grinding out increments for hours before hitting a wall would waste your time and mine, and I'd end up reporting a number like 4,312 as if it were progress toward a goal I already knew was unreachable.

The counter also gives me no way to shortcut: it only does +1 and returns the new value. There's no set, no add-N, no read-without-increment. So there's no clever path to 1,000,000 either.

Here's what I can actually do:

- **Demonstrate the tool works** — a handful of calls so you can see the counter's behavior and starting point.
- **Increment to a reachable ta

## 5. Give it a tool it doesn't need

Add one irrelevant tool alongside the real ones. Confirm the model doesn't call it just because it's available.

In [6]:
irrelevant_tool = {
    "name": "get_weather",
    "description": "Get the current weather for a city. Not related to prices or arithmetic.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
}

def run_weather(inp):
    return "72F and sunny"

tools_with_distractor = [calculator_tool, lookup_tool, irrelevant_tool]
functions_with_distractor = dict(TOOL_FUNCTIONS)
functions_with_distractor["get_weather"] = run_weather

answer = react_loop_basic(
    "What is the total cost of 4 of widget_c?",
    tools_with_distractor, functions_with_distractor,
)
print(answer)
print('\n(If this printed a real cost and never mentioned weather, the model correctly ignored the irrelevant tool.)')

The total cost of 4 widget_c units is **$32.00** (4 × $8.00).

(If this printed a real cost and never mentioned weather, the model correctly ignored the irrelevant tool.)


## 6. Track cost across the whole loop

Reuse Phase 3's cost-tracking wrapper. Sum it across every turn in the loop, not just one call.

In [7]:
INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

def react_loop_metered(user_message, tools, tool_functions, max_tokens=500, max_steps=8):
    messages = [{"role": "user", "content": user_message}]
    step = 0
    total_cost = 0.0

    while step < max_steps:
        step += 1
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
        total_cost += call_cost(response.usage)

        if response.stop_reason != 'tool_use':
            final_text = ''.join(b.text for b in response.content if b.type == 'text')
            return final_text, step, total_cost

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                result = tool_functions[block.name](block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})

    return None, step, total_cost

answer, steps_used, total_cost = react_loop_metered(
    "Look up the price of widget_a, widget_b, and widget_c, then tell me their combined total.",
    TOOLS, TOOL_FUNCTIONS,
)

print(f'Answer: {answer}')
print(f'Steps used: {steps_used}')
print(f'Total cost across the loop: ${total_cost:.6f}')

Answer: Here are the prices:

| Product | Price |
|---|---|
| widget_a | $42.50 |
| widget_b | $17.25 |
| widget_c | $8.00 |
| **Total** | **$67.75** |

The combined total is **$67.75**.
Steps used: 3
Total cost across the loop: $0.019850
